# 📊 Exploratory Data Analysis — Olist Brazilian E-Commerce

## Business Context
> "We're growing fast but our margins are getting squeezed. Sellers are complaining, buyers leave bad reviews, and we don't really know where to focus."

This notebook documents the data exploration that led to choosing **Late Delivery Prediction** as the highest-value ML intervention.

**Dataset**: ~100K orders across 9 linked tables from a Brazilian marketplace (2016–2018).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Download data
DATA_DIR = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print(f"Data directory: {DATA_DIR}")

## 1. Load & Profile All Tables

In [ ]:
import os

orders = pd.read_csv(os.path.join(DATA_DIR, 'olist_orders_dataset.csv'))
items = pd.read_csv(os.path.join(DATA_DIR, 'olist_order_items_dataset.csv'))
payments = pd.read_csv(os.path.join(DATA_DIR, 'olist_order_payments_dataset.csv'))
reviews = pd.read_csv(os.path.join(DATA_DIR, 'olist_order_reviews_dataset.csv'))
customers = pd.read_csv(os.path.join(DATA_DIR, 'olist_customers_dataset.csv'))
sellers = pd.read_csv(os.path.join(DATA_DIR, 'olist_sellers_dataset.csv'))
products = pd.read_csv(os.path.join(DATA_DIR, 'olist_products_dataset.csv'))
translation = pd.read_csv(os.path.join(DATA_DIR, 'product_category_name_translation.csv'))
products = products.merge(translation, on='product_category_name', how='left')

# Parse dates
for col in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
            'order_delivered_customer_date', 'order_estimated_delivery_date']:
    orders[col] = pd.to_datetime(orders[col])

tables = {
    'orders': orders, 'items': items, 'payments': payments,
    'reviews': reviews, 'customers': customers, 'sellers': sellers,
    'products': products
}

for name, df in tables.items():
    print(f"\n{'='*50}")
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"Nulls: {df.isnull().sum()[df.isnull().sum() > 0].to_dict() or 'None'}")

## 2. Order Status Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Status counts
status = orders['order_status'].value_counts()
ax = axes[0]
bars = ax.barh(status.index, status.values, color=sns.color_palette('husl', len(status)))
ax.set_title('Order Status Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Count')
for bar, val in zip(bars, status.values):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center')

# Monthly order volume
orders_ts = orders.set_index('order_purchase_timestamp').resample('ME').size()
ax = axes[1]
ax.plot(orders_ts.index, orders_ts.values, 'o-', linewidth=2, markersize=4)
ax.set_title('Monthly Order Volume', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('# Orders')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
print(f"\nDate range: {orders['order_purchase_timestamp'].min()} → {orders['order_purchase_timestamp'].max()}")
print(f"Total orders: {len(orders):,}")
print(f"Delivered: {(orders['order_status'] == 'delivered').sum():,} ({(orders['order_status'] == 'delivered').mean()*100:.1f}%)")

## 3. Review Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
colors = ['#d32f2f', '#f57c00', '#fbc02d', '#7cb342', '#388e3c']
ax = axes[0]
score_counts = reviews['review_score'].value_counts().sort_index()
ax.bar(score_counts.index, score_counts.values, color=colors)
ax.set_title('Review Score Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Review Score')
ax.set_ylabel('Count')
for i, (idx, val) in enumerate(score_counts.items()):
    ax.text(idx, val + 500, f'{val:,}', ha='center', fontweight='bold')

# Bad reviews text analysis
ax = axes[1]
reviews_text = reviews[reviews['review_comment_message'].notna()].copy()
bad = reviews_text[reviews_text['review_score'] <= 2].copy()
bad['msg'] = bad['review_comment_message'].str.lower()

categories = {
    'Delivery/Delay': ['entrega','atraso','atrasa','demor','prazo','chegou','chegar'],
    'Product Quality': ['defeito','qualidade','quebrado','veio errado','danificado','diferente'],
    'Communication': ['resposta','vendedor','comunica','contato','retorno','atendimento']
}
cat_counts = {}
for cat, words in categories.items():
    mask = bad['msg'].apply(lambda x: any(w in str(x) for w in words))
    cat_counts[cat] = mask.sum()

ax.barh(list(cat_counts.keys()), list(cat_counts.values()),
        color=['#e53935', '#fb8c00', '#43a047'])
ax.set_title(f'What Bad Reviews (1-2★) Mention (n={len(bad):,})', fontsize=14, fontweight='bold')
for i, (cat, val) in enumerate(cat_counts.items()):
    ax.text(val + 50, i, f'{val:,} ({val/len(bad)*100:.0f}%)', va='center')

plt.tight_layout()
plt.show()

print(f"\nBad reviews (1-2 stars): {(reviews['review_score'] <= 2).sum():,} "
      f"({(reviews['review_score'] <= 2).sum()/len(reviews)*100:.1f}%)")
print(f"Reviews with text: {reviews['review_comment_message'].notna().sum():,} "
      f"({reviews['review_comment_message'].notna().mean()*100:.0f}%)")

## 4. 🔑 The Key Insight: Late Delivery → Bad Reviews

This is the analysis that drove our problem selection. We compute delivery delay and correlate it with review scores.

In [ ]:
delivered = orders[orders['order_status'] == 'delivered'].dropna(subset=['order_delivered_customer_date']).copy()
delivered['delay_days'] = (delivered['order_delivered_customer_date'] -
                           delivered['order_estimated_delivery_date']).dt.total_seconds() / 86400
delivered['actual_days'] = (delivered['order_delivered_customer_date'] -
                            delivered['order_purchase_timestamp']).dt.total_seconds() / 86400
delivered['is_late'] = delivered['delay_days'] > 0

print(f"Total delivered orders: {len(delivered):,}")
print(f"Late deliveries: {delivered['is_late'].sum():,} ({delivered['is_late'].mean()*100:.1f}%)")
print(f"\nDelay stats (when late):")
print(delivered[delivered['is_late']]['delay_days'].describe().round(1))

In [ ]:
# THE KILLER CHART: Late delivery vs review score
merged = delivered.merge(reviews[['order_id', 'review_score']], on='order_id', how='inner')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Mean review by delivery status
ax = axes[0]
means = merged.groupby('is_late')['review_score'].mean()
bars = ax.bar(['On Time', 'Late'], means.values, color=['#388e3c', '#d32f2f'], width=0.5)
ax.set_title('Mean Review Score', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Review (1-5)')
ax.set_ylim(1, 5)
for bar, val in zip(bars, means.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.1, f'{val:.2f}', ha='center', fontsize=16, fontweight='bold')

# Chart 2: Review distribution for late orders
ax = axes[1]
late_reviews = merged[merged['is_late']]['review_score'].value_counts().sort_index()
ax.bar(late_reviews.index, late_reviews.values, color=colors)
ax.set_title('Review Distribution (Late Orders)', fontsize=14, fontweight='bold')
ax.set_xlabel('Review Score')
ax.set_ylabel('Count')
for idx, val in late_reviews.items():
    ax.text(idx, val + 20, f'{val/late_reviews.sum()*100:.0f}%', ha='center', fontweight='bold')

# Chart 3: Delay vs review (scatter density)
ax = axes[2]
late = merged[merged['is_late']].copy()
ax.hexbin(late['delay_days'].clip(0, 50), late['review_score'],
          gridsize=20, cmap='YlOrRd', mincnt=1)
ax.set_title('Delay (days) vs Review Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Days Late')
ax.set_ylabel('Review Score')
plt.colorbar(ax.collections[0], ax=ax, label='Count')

plt.tight_layout()
plt.show()

print(f"\n⚡ KEY FINDING:")
print(f"   On-time mean review: {means[False]:.2f}")
print(f"   Late mean review:    {means[True]:.2f}")
print(f"   Delta:               {means[False] - means[True]:.2f} stars")
print(f"   45% of late orders get 1-star reviews")

## 5. Geographic Analysis: Cross-State = More Risk

In [ ]:
# Join customer & seller location
full = delivered.merge(customers[['customer_id', 'customer_state']], on='customer_id')
full = full.merge(items[['order_id', 'seller_id', 'price', 'freight_value']], on='order_id')
full = full.merge(sellers[['seller_id', 'seller_state']], on='seller_id')
full['same_state'] = full['customer_state'] == full['seller_state']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cross-state vs same-state
ax = axes[0]
geo = full.groupby('same_state').agg(
    count=('order_id', 'count'),
    late_rate=('is_late', 'mean'),
    avg_delivery=('actual_days', 'mean')
)
bars = ax.bar(['Cross-State', 'Same-State'], geo['late_rate'].values * 100,
              color=['#e53935', '#43a047'], width=0.5)
ax.set_title('Late Delivery Rate by Geography', fontsize=14, fontweight='bold')
ax.set_ylabel('% Late')
for bar, val, count in zip(bars, geo['late_rate'].values, geo['count'].values):
    ax.text(bar.get_x() + bar.get_width()/2, val*100 + 0.3,
            f'{val*100:.1f}%\n(n={count:,})', ha='center', fontweight='bold')

# Top states by late rate
ax = axes[1]
state_late = full.groupby('customer_state').agg(
    orders=('order_id', 'nunique'),
    late_rate=('is_late', 'mean')
).query('orders > 200').sort_values('late_rate', ascending=True)
ax.barh(state_late.index, state_late['late_rate'] * 100,
        color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(state_late))))
ax.set_title('Late Delivery Rate by Customer State', fontsize=14, fontweight='bold')
ax.set_xlabel('% Late')
ax.axvline(x=full['is_late'].mean()*100, color='black', linestyle='--', alpha=0.5, label='Overall')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nCross-state orders: {(~full['same_state']).sum():,} ({(~full['same_state']).mean()*100:.0f}%)")
print(f"Same-state orders:  {full['same_state'].sum():,}")

## 6. Freight as a Margin Pressure Signal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Freight % of price
items_clean = items[items['price'] > 0].copy()
items_clean['freight_pct'] = items_clean['freight_value'] / items_clean['price'] * 100

ax = axes[0]
ax.hist(items_clean['freight_pct'].clip(0, 100), bins=50, color='#5c6bc0', alpha=0.8, edgecolor='white')
ax.axvline(x=items_clean['freight_pct'].median(), color='red', linestyle='--',
           label=f'Median: {items_clean["freight_pct"].median():.0f}%')
ax.set_title('Freight as % of Item Price', fontsize=14, fontweight='bold')
ax.set_xlabel('Freight / Price (%)')
ax.set_ylabel('Count')
ax.legend()

# Price distribution
ax = axes[1]
ax.hist(items['price'].clip(0, 500), bins=50, color='#26a69a', alpha=0.8, edgecolor='white')
ax.axvline(x=items['price'].median(), color='red', linestyle='--',
           label=f'Median: R${items["price"].median():.0f}')
ax.set_title('Item Price Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Price (R$)')
ax.set_ylabel('Count')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Freight as % of price: median={items_clean['freight_pct'].median():.1f}%, mean={items_clean['freight_pct'].mean():.1f}%")
print(f"→ Shipping cost is a MAJOR margin factor")

## 7. Why We Ruled Out Alternatives

### Repeat Customers → Too Sparse for Churn Model

In [ ]:
cust_orders = customers.merge(orders[['customer_id', 'order_id']], on='customer_id')
repeat = cust_orders.groupby('customer_unique_id')['order_id'].nunique()
print(f"Total unique customers: {len(repeat):,}")
print(f"Repeat customers (>1 order): {(repeat > 1).sum():,} ({(repeat > 1).sum()/len(repeat)*100:.1f}%)")
print(f"→ Only 3.1% repeat rate — not enough signal for churn prediction")
print(f"\nCancellation rate: {(orders['order_status'] == 'canceled').sum()} / {len(orders)} = "
      f"{(orders['order_status'] == 'canceled').mean()*100:.2f}%")
print(f"→ Too low for fraud detection")

# Seller quality
seller_perf = full.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
seller_avg = seller_perf.groupby('seller_id').agg(
    orders=('order_id', 'nunique'),
    avg_review=('review_score', 'mean')
).query('orders >= 10')
print(f"\nSellers with avg review < 3: {(seller_avg['avg_review'] < 3).sum()} / {len(seller_avg)}")
print(f"→ The problem isn't bad sellers — it's logistics")

## 8. Conclusion: Why Late Delivery Prediction

| Signal | Evidence | Impact |
|--------|----------|--------|
| Late delivery → bad reviews | Mean review drops from 4.29 → 2.57 | Direct NPS/retention hit |
| Cross-state = risky | 9.0% late vs 5.9% same-state | 64% of orders are cross-state |
| Freight squeezes margins | 23-32% of item price | Late = refunds on top |
| Reviews mention delivery | 33% of bad reviews | Customers notice |
| Repeat rate is tiny | 3.1% | Bad experience = lost customer |

**Late delivery is the single intervention point that touches all three customer pain points: margins, seller friction, and buyer dissatisfaction.**

The model predicts late delivery at order time — giving the ops team a concrete lever to flag orders for expedited handling or proactive customer notification.